# contextlib

对应 `stdlib.md`：写 `with`：进入、退出、可选资源和批量嵌套。

笔记本在 `python_stdlibs/contextlib/qa.ipynb`。需要读写文件时，工作空间就是这个目录。

先运行下一格，得到 `ROOT`。每题只改 `# 作答` 下面的代码。前置代码不用改。做完自己跑通即可，先不要对答案。

`contextlib` 管的是 `with` 的进入和退出。它不是 `contextvars`（那个存任务局部变量）。


In [1]:
from pathlib import Path

def lab_root() -> Path:
    """qa.ipynb 所在目录，即 python_stdlibs/contextlib。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "contextlib" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "python_stdlibs" / "contextlib"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here

ROOT = lab_root()
ROOT

PosixPath('/Users/keyficller/Documents/AEFS-Notes/codes/python_stdlibs/contextlib')

## 1. 手写进入和退出

写一个类 `Note`。进入 `with` 时打印 `enter` 并交出 `"open"`，离开时打印 `exit`。用它包住一行 `print`，把交出的值打出来。


In [2]:
# 作答

class Note:
    def __enter__(self):
        print("enter")
        return "open"

    def __exit__(self, exc_type, exc, tb):
        print("exit")
        return False

with Note() as value:
    print(value)
#评阅
# 对。__enter__ 的返回值交给 as，__exit__ 返回 False，异常不会被吃掉。

#参考答案
# class Note:
#     def __enter__(self):
#         print("enter")
#         return "open"
#     def __exit__(self, exc_type, exc, tb):
#         print("exit")
#         return False
# with Note() as value:
#     print(value)

enter
open
exit


## 2. 用生成器写同一件事

用 `@contextmanager` 写 `note()`，打印顺序和上一题一样：`enter`、交出的 `"open"`、`exit`。清理放在 `finally` 里。


In [3]:
# 作答

from contextlib import contextmanager

@contextmanager
def note():
    print("enter")
    try:
        yield "open"
    finally:
        print("exit")

with note() as value:
    print(value)
#评阅
# 对。yield 之前是进入，yield 的值交给 as，finally 保证抛异常时也会打印 exit。

#参考答案
# from contextlib import contextmanager
# @contextmanager
# def note():
#     print("enter")
#     try:
#         yield "open"
#     finally:
#         print("exit")
# with note() as value:
#     print(value)

enter
open
exit


## 3. 只吞一种异常

读 `ROOT / "missing.txt"`。文件不存在时不要让程序停，打印 `ok`。其它异常仍要抛出来。


In [4]:
from pathlib import Path

path = ROOT / "missing.txt"

# 作答

from contextlib import suppress

with suppress(FileNotFoundError):
    Path(path).read_text()
print("ok")
#评阅
# 对。suppress 只吃列出的类型。这里文件确实不存在，所以走到 print。

#参考答案
# from contextlib import suppress
# with suppress(FileNotFoundError):
#     Path(path).read_text()
# print("ok")

ok


## 4. 给没有 with 的对象收尾

`Handle` 只有 `close()`，没有 `__enter__`。用 `with` 包住它：里面打印 `using`，离开时必须打印 `closed`。


In [5]:
class Handle:
    def close(self) -> None:
        print("closed")

# 作答

from contextlib import closing

with closing(Handle()):
    print("using")
#评阅
# 对。closing 在离开时调用 close()，对象自己不用实现 with。

#参考答案
# from contextlib import closing
# with closing(Handle()):
#     print("using")

using
closed


## 5. 有时没有资源

`use(lock)` 在 `lock` 不是 `None` 时用它做 `with`，是 `None` 时仍走同一条 `with`，`as` 拿到 `"none"`。调用 `use(None)` 并打印这个值。


In [6]:
# 作答

from contextlib import nullcontext

def use(lock):
    cm = lock if lock is not None else nullcontext("none")
    with cm as got:
        print(got)

use(None)
#评阅
# 对。nullcontext 什么都不做，只把 enter_result 交给 as，这样分支不用复制 with 里面的代码。

#参考答案
# from contextlib import nullcontext
# def use(lock):
#     cm = lock if lock is not None else nullcontext("none")
#     with cm as got:
#         print(got)
# use(None)

none


## 6. 个数运行时才知道

`tag(name)` 已经能 `with`。名字列表是 `["a", "b"]`，运行时才知道有几个。一次进入全部，里面打印 `a,b`。离开时后进的先退出。


In [7]:
from contextlib import contextmanager

@contextmanager
def tag(name: str):
    print(f"open {name}")
    try:
        yield name
    finally:
        print(f"close {name}")

names = ["a", "b"]

# 作答

from contextlib import ExitStack

with ExitStack() as stack:
    opened = [stack.enter_context(tag(name)) for name in names]
    print(",".join(opened))
#评阅
# 对。enter_context 立刻进入并登记退出。栈是后进先出，所以先 close b 再 close a。

#参考答案
# from contextlib import ExitStack
# with ExitStack() as stack:
#     opened = [stack.enter_context(tag(name)) for name in names]
#     print(",".join(opened))

open a
open b
a,b
close b
close a
